In [1]:
import os
from dotenv import load_dotenv,find_dotenv

_ = load_dotenv(find_dotenv(),override=True)

openai_key = os.getenv("OPENAI_API_KEY")

In [2]:
from langgraph.graph import StateGraph,END
from  typing import TypedDict, Annotated
import operator
from  langchain_core.messages import AnyMessage,SystemMessage,HumanMessage,ToolMessage
from  langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()


C:\Users\anura\AppData\Local\Temp\ipykernel_20572\926394804.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults


In [3]:
from uuid import uuid4

In [4]:
def reduce_messages(left: list[AnyMessage],right: list[AnyMessage])-> list[AnyMessage]:
    for message in right:
        if not message.id:
            message.id = str(uuid4())

    merged = left.copy()
    for message in right:
        for i , existing in enumerate(merged):
            if existing.id == message.id:
                merged[i] = message
                break
        else:
            merged.append(message)
    return merged


class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage],reduce_messages]









In [5]:
tool = TavilySearchResults(max_results=2)

C:\Users\anura\AppData\Local\Temp\ipykernel_20572\4289725543.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tool = TavilySearchResults(max_results=2)


In [6]:
class Agent:
    def __init__(self, model, tools, checkpointer = None, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile(checkpointer=checkpointer,
                                   interrupt_before=["action"])
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def call_openai(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [7]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""
model = ChatOpenAI(model = "openai/gpt-3.5-turbo",base_url="https://openrouter.ai/api/v1",api_key=openai_key)
#with SqliteSaver.from_conn_string(":memory:") as memory:
abot = Agent(model, [tool], system=prompt, checkpointer=memory)

In [8]:
messages = [HumanMessage(content="Whats the weather in SF?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 152, 'total_tokens': 174, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.000109, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.000109, 'upstream_inference_prompt_cost': 7.6e-05, 'upstream_inference_completions_cost': 3.3e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-3.5-turbo', 'system_fingerprint': None, 'id': 'gen-1780800665-wo96oqRLPthOgZJSit9d', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e9ffd-b467-78d2-bb00-5630929d2210-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'current weather in San Francisco'}, 'id': 'call_qqdtsUs

In [9]:
abot.graph.get_state(thread)

StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in SF?', additional_kwargs={}, response_metadata={}, id='71d9bf73-0a00-4f70-bf7c-b341bd4fd277'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 152, 'total_tokens': 174, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.000109, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.000109, 'upstream_inference_prompt_cost': 7.6e-05, 'upstream_inference_completions_cost': 3.3e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-3.5-turbo', 'system_fingerprint': None, 'id': 'gen-1780800665-wo96oqRLPthOgZJSit9d', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e9ffd

In [10]:
abot.graph.get_state(thread).next

('action',)

In [11]:
for event in abot.graph.stream(None,thread):
    for v in event.values():
        print(v)

Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'current weather in San Francisco'}, 'id': 'call_qqdtsUsAChNAAB1ePCk58Btw', 'type': 'tool_call'}
Back to the model!
{'messages': [ToolMessage(content='[{\'title\': \'San Francisco, CA Monthly Weather - AccuWeather\', \'url\': \'https://www.accuweather.com/en/us/san-francisco/94103/june-weather/347629\', \'content\': "San Francisco\'s June 2026 forecast shows daily high temperatures ranging from 63° to 79°, with overnight lows between 50° and 58°. The average high for June is", \'score\': 0.99985456}, {\'title\': \'Weather San Francisco in June 2026: Temperature & Climate\', \'url\': \'https://en.climate-data.org/north-america/united-states-of-america/california/san-francisco-385/t/june-6\', \'content\': \'Daily precipitation in San Francisco during June ranges from 0.02/0.00 mm/in to 1.29/0.05 mm/in.\\nThe rainiest day is 04.06, with 1.29/0.05 mm/in, while the driest day is 20.06, bringing only 0.02/0.00 mm/in.\\nThe fir

In [12]:
abot.graph.get_state(thread)


StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in SF?', additional_kwargs={}, response_metadata={}, id='71d9bf73-0a00-4f70-bf7c-b341bd4fd277'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 152, 'total_tokens': 174, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.000109, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.000109, 'upstream_inference_prompt_cost': 7.6e-05, 'upstream_inference_completions_cost': 3.3e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-3.5-turbo', 'system_fingerprint': None, 'id': 'gen-1780800665-wo96oqRLPthOgZJSit9d', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e9ffd

In [13]:
abot.graph.get_state(thread).next

()

In [14]:
messages = [HumanMessage("what is the weather in LA")]
thread = {"configurable":{"thread_id":"2"}}
for event in abot.graph.stream({"messages":messages},thread):
    for v in event.values():
        print(v)

while abot.graph.get_state(thread).next:
    print(abot.graph.get_state(thread))
    _input = input("Proceed????????????")

    if _input != "y":
        print("aborting")
        break

    for event in abot.graph.stream(None, thread):
        for v in event.values():
            print(v)

{'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 152, 'total_tokens': 172, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.000106, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.000106, 'upstream_inference_prompt_cost': 7.6e-05, 'upstream_inference_completions_cost': 3e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-3.5-turbo', 'system_fingerprint': None, 'id': 'gen-1780800671-2hhPJYZgpZ9iTbqCyoM9', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e9ffd-cc54-7d53-b2ad-077c374f2760-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'Los Angeles weather'}, 'id': 'call_Z18tLIcu0MNfc6nijwboJa

In [15]:
messages = [HumanMessage("What is the weather in LA?")]
thread = {"configurable":{"thread_id":"3"}}
for event in abot.graph.stream({"messages":messages},thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 153, 'total_tokens': 174, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.000108, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.000108, 'upstream_inference_prompt_cost': 7.65e-05, 'upstream_inference_completions_cost': 3.15e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-3.5-turbo', 'system_fingerprint': None, 'id': 'gen-1780800688-tsNg9qTN2CG2bcQq1ZM4', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e9ffe-0f91-71f3-8e19-942bd944bfb1-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'weather in Los Angeles'}, 'id': 'call_ag88cvO8d1z3sFF

In [16]:
abot.graph.get_state(thread)

StateSnapshot(values={'messages': [HumanMessage(content='What is the weather in LA?', additional_kwargs={}, response_metadata={}, id='3b25d8cd-6b5d-49f5-a5ac-6f491bb4fe5a'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 153, 'total_tokens': 174, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.000108, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.000108, 'upstream_inference_prompt_cost': 7.65e-05, 'upstream_inference_completions_cost': 3.15e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-3.5-turbo', 'system_fingerprint': None, 'id': 'gen-1780800688-tsNg9qTN2CG2bcQq1ZM4', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e

In [17]:
current_values = abot.graph.get_state(thread)

In [18]:
current_values.values['messages'][-1]

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 153, 'total_tokens': 174, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.000108, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.000108, 'upstream_inference_prompt_cost': 7.65e-05, 'upstream_inference_completions_cost': 3.15e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-3.5-turbo', 'system_fingerprint': None, 'id': 'gen-1780800688-tsNg9qTN2CG2bcQq1ZM4', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e9ffe-0f91-71f3-8e19-942bd944bfb1-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'weather in Los Angeles'}, 'id': 'call_ag88cvO8d1z3sFFqLcAyv2Ny', 't

In [19]:
current_values.values['messages'][-1].tool_calls

[{'name': 'tavily_search_results_json',
  'args': {'query': 'weather in Los Angeles'},
  'id': 'call_ag88cvO8d1z3sFFqLcAyv2Ny',
  'type': 'tool_call'}]

In [20]:
_id = current_values.values['messages'][-1].tool_calls[0]['id']

current_values.values['messages'][-1].tool_calls = [
    {'name': 'tavily_search_results_json',
     'args': {'query': 'current weather in Louisiana'},
     'id': _id}
]



In [21]:
abot.graph.update_state(thread,current_values.values)

{'configurable': {'thread_id': '3',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1621bc-83a4-6363-8002-262dc1a62b44'}}

In [22]:
abot.graph.get_state(thread)

StateSnapshot(values={'messages': [HumanMessage(content='What is the weather in LA?', additional_kwargs={}, response_metadata={}, id='3b25d8cd-6b5d-49f5-a5ac-6f491bb4fe5a'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 153, 'total_tokens': 174, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.000108, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.000108, 'upstream_inference_prompt_cost': 7.65e-05, 'upstream_inference_completions_cost': 3.15e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-3.5-turbo', 'system_fingerprint': None, 'id': 'gen-1780800688-tsNg9qTN2CG2bcQq1ZM4', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e

In [23]:
for event in abot.graph.stream(None, thread):
    for v in event.values():
        print(v)

Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'current weather in Louisiana'}, 'id': 'call_ag88cvO8d1z3sFFqLcAyv2Ny', 'type': 'tool_call'}
Back to the model!
{'messages': [ToolMessage(content='[{\'title\': \'Louisiana weather in June 2026 | Louisiana 14 day weather\', \'url\': \'https://www.weather25.com/north-america/usa/louisiana?page=month&month=June\', \'content\': \'| Month | Temperatures | Rainy Days | Dry Days | Snowy Days | Rainfall | Weather | More details |\\n ---  ---  ---  --- |\\n| January | 16° / 9° | 4 | 27 | 0 | 53 mm | Good | Louisiana in January |\\n| February | 19° / 12° | 4 | 24 | 0 | 60 mm | Good | Louisiana in February |\\n| March | 22° / 15° | 4 | 27 | 0 | 65 mm | Good | Louisiana in March |\\n| April | 25° / 18° | 4 | 26 | 0 | 56 mm | Perfect | Louisiana in April |\\n| May | 29° / 21° | 7 | 24 | 0 | 85 mm | Ok | Louisiana in May |\\n| June | 32° / 25° | 10 | 20 | 0 | 136 mm | Ok | Louisiana in June |\\n| July | 32° / 26° | 13 | 18 | 0 | 189 m

In [24]:
states =[]
for state in abot.graph.get_state_history(thread):
    print(state)
    print('--##################---------')
    states.append(state)

StateSnapshot(values={'messages': [HumanMessage(content='What is the weather in LA?', additional_kwargs={}, response_metadata={}, id='3b25d8cd-6b5d-49f5-a5ac-6f491bb4fe5a'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 153, 'total_tokens': 174, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.000108, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.000108, 'upstream_inference_prompt_cost': 7.65e-05, 'upstream_inference_completions_cost': 3.15e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-3.5-turbo', 'system_fingerprint': None, 'id': 'gen-1780800688-tsNg9qTN2CG2bcQq1ZM4', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e

In [25]:
to_replay = states[-3]

In [26]:
to_replay

StateSnapshot(values={'messages': [HumanMessage(content='What is the weather in LA?', additional_kwargs={}, response_metadata={}, id='3b25d8cd-6b5d-49f5-a5ac-6f491bb4fe5a'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 153, 'total_tokens': 174, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.000108, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.000108, 'upstream_inference_prompt_cost': 7.65e-05, 'upstream_inference_completions_cost': 3.15e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-3.5-turbo', 'system_fingerprint': None, 'id': 'gen-1780800688-tsNg9qTN2CG2bcQq1ZM4', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e

In [27]:
for event in abot.graph.stream(None,to_replay.config):
    for k,v in event.items():
        print(v)

Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'weather in Los Angeles'}, 'id': 'call_ag88cvO8d1z3sFFqLcAyv2Ny', 'type': 'tool_call'}
Back to the model!
{'messages': [ToolMessage(content='[{\'title\': \'Weather La California in June 2026: Temperature & Climate\', \'url\': \'https://en.climate-data.org/europe/italy/tuscany/la-california-274344/t/june-6\', \'content\': \'In June, La California sees average air temperatures of 21.7°C | 71.1°F, while water temperatures increase from 20°C | 68°F at the start of the month to 21.3°C | 70.3°F mid-month and 22.6°C | 72.7°F by the end.\\n\\nDuring June, La California enjoys anywhere from 12 hours to 13 hours of sun each day.\\n27.06 is the sunniest day with 13 hours, while 06.06 sees the least sun with just 12 hours.\\nSunshine averages 12 hours in the early days, climbs to 13 hours mid-month, and settles at 13 hours near the end of the month.\\nThe brightest 10-day period runs from 21-30, with daily values of 13 hours.\\nThe 

In [28]:
_id = to_replay.values['messages'][-1].tool_calls[0]['id']
to_replay.values['messages'][-1].tool_calls = [{'name': 'tavily_search_results_json',
  'args': {'query': 'current weather in LA, accuweather'},
  'id': _id}]

In [29]:
branch_state = abot.graph.update_state(to_replay.config,to_replay.values)

In [30]:
for event in abot.graph.stream(None,branch_state):
    for k,v in event.items():
        print(v)

Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'current weather in LA, accuweather'}, 'id': 'call_ag88cvO8d1z3sFFqLcAyv2Ny', 'type': 'tool_call'}
Back to the model!
{'messages': [ToolMessage(content='[{\'title\': \'Los Angeles, CA Monthly Weather | AccuWeather\', \'url\': \'https://www.accuweather.com/en/us/los-angeles/90012/july-weather/347625\', \'content\': "### Winter Center\\n\\n## Monthly\\n\\n## July\\n\\n## 2026\\n\\n## 10-Day\\n\\nMostly sunny\\nConsiderable cloudiness\\nMostly sunny\\nPlenty of sunshine\\nMostly sunny\\nMostly sunny\\nPlenty of sunshine\\nPlenty of sunshine\\nPlenty of sunshine\\nPlenty of sun\\nPlenty of sun\\nPlenty of sun\\nPlenty of sunshine\\nPlenty of sunshine\\nPlenty of sunshine\\nMostly sunny\\nMore sun than clouds\\nPartly sunny\\nSunny\\nMostly sunny\\nMostly sunny\\nPartly sunny\\nSunny to partly cloudy\\nMostly sunny\\nPartly sunny\\nMostly cloudy\\nPartly sunny\\nSunshine and a few clouds\\nSome sun\\nMostly sunny\\nMostly sun

In [31]:
to_replay

StateSnapshot(values={'messages': [HumanMessage(content='What is the weather in LA?', additional_kwargs={}, response_metadata={}, id='3b25d8cd-6b5d-49f5-a5ac-6f491bb4fe5a'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 153, 'total_tokens': 174, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.000108, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.000108, 'upstream_inference_prompt_cost': 7.65e-05, 'upstream_inference_completions_cost': 3.15e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-3.5-turbo', 'system_fingerprint': None, 'id': 'gen-1780800688-tsNg9qTN2CG2bcQq1ZM4', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e

In [32]:
_id = to_replay.values['messages'][-1].tool_calls[0]['id']


In [33]:
state_update = {"messages":[ToolMessage(tool_call_id=_id,name="tavily_search_results_json",content="54 degree celcius")]}



In [34]:
branch_as_node = abot.graph.update_state(to_replay.config,state_update,as_node="action")

In [35]:
for event in abot.graph.stream(None,branch_as_node):
    for k,v in event.items():
        print(v)

{'messages': [AIMessage(content='The current weather in Los Angeles is 54 degrees Celsius.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 191, 'total_tokens': 204, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.000115, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.000115, 'upstream_inference_prompt_cost': 9.55e-05, 'upstream_inference_completions_cost': 1.95e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-3.5-turbo', 'system_fingerprint': None, 'id': 'gen-1780800706-EOFbDmrFuRH1T5gi1CmL', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e9ffe-5513-75d3-b238-10c97d7c19da-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 19